# Dana et al. (2016), red calcium indicators: Figure 4 on CRCNS cai-2

Paper: Dana, Mohar, Sun et al. (2016), "Sensitive red protein calcium indicators for imaging
neural activity", eLife 5:e12727, doi:10.7554/elife.12727. The claims file
(`claims/dana-2016.yaml`) was written from the version of record (PMC4846379) with its Methods,
figure legends and Supplementary files 1 and 2.

Dataset: CRCNS **cai-2** (doi:10.6080/K0W37T87), read anonymously from the public S3 bucket
`crcnsarchive` under `cai-2/live/`. CRCNS data sets are not versioned, so the version is recorded
as `unversioned` (docs/archives.md). `cai-2/live/` is the only published copy of the data (the
bucket also has `cai-2/orig/`, which holds only an older description document), so it is the one
pinned. The data set's `checksums.md5` gives `e0bdb7e619b82a99fbe594ef62f647cf` for
`data/jRGECO1a_data.tar.gz` and `d29a21d0d4a40ebc774ca9edb1079228` for `data/jRCAMP1a_data.tar.gz`;
the data check below computes the MD5 of each archive as it streams it and compares the two, so a
later change to the files can be seen. The data set's description is version 0.8 (April 19, 2016).

Outside inputs: Supplementary files 1 and 2 of the paper (doi:10.7554/eLife.12727.030 and .031),
streamed from the eLife CDN, for the three `data: elsewhere` claims on the culture screen.

The data set holds only the simultaneous imaging and cell-attached recordings of Figure 4. Claims
on the paper's other experiments are blocked, as the record says.

| Claim | Tested here |
|---|---|
| screen-single-mutations-improved | tested, on Supplementary files 1 and 2 (elsewhere) |
| culture-jrgeco1a-sensitivity | tested, on Supplementary file 2 (elsewhere) |
| culture-jrcamp1-sensitivity | tested, on Supplementary file 1 (elsewhere) |
| fig4-recorded-cells | tested |
| fig4d-event-counts | tested |
| fig4e-jrgeco1a-1ap-detection | tested |
| fig4e-jrgeco1a-2ap-detection | tested |
| fig4e-jrcamp1a-detection | tested |
| fig4df-jrgeco1a-better-than-jrcamp1a | tested |
| fig4g-jrgeco1a-faster | tested |
| fig4g-kinetics-like-gcamp6 | blocked: the GCaMP6 values of Chen et al. (2013b) were not obtained in a form the notebook can read |
| culture, purified protein, V1 grating, Drosophila, zebrafish and C. elegans claims (24) | blocked: data missing (not in cai-2, not deposited elsewhere) |

Method details that the claims file does not give are taken from the paper's Methods or from the
data set's own analysis script `data_analysis_for_CRCNS_jRGECO1a.m` (with `find_no_event.m` and
`find_nspike_event.m`, in `cai-2/live/scripts/`), which the data description says produced the
Figure 4 data. Each is cited in `cai2.py`.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import io, re, time, resource, itertools
import numpy as np
import pandas as pd
import yaml
import fsspec
import matplotlib.pyplot as plt
import cai2

DATASET = "crcns:cai-2/unversioned"   # CRCNS data set cai-2, doi:10.6080/K0W37T87, not versioned
T0 = time.time()
CLAIMS = {c["id"]: c for c in yaml.safe_load(open("../../claims/dana-2016.yaml"))["claims"]}

def expected(cid):
    e = CLAIMS[cid].get("expected", [])
    return e if isinstance(e, list) else [e]

def within(value, e):
    return abs(value - e["value"]) <= e["tolerance"] + 1e-9

## Data check

The bucket's listing, the published checksums, and every recording file in the two archives. The
archives are streamed and decompressed as they are read; each MATLAB file is reduced to the soma
and neuropil fluorescence, the frame times and the spike times, and the 10 kHz ephys traces are
dropped after their range and hash are kept.

In [ ]:
with fsspec.open("https://crcnsarchive.s3-us-west-2.amazonaws.com/?list-type=2&prefix=cai-2/", "rt") as f:
    listing = f.read()
keys = re.findall(r"<Key>(.*?)</Key>\s*<LastModified>(.*?)</LastModified>.*?<Size>(\d+)</Size>", listing, re.S)
print("truncated listing:", re.findall(r"<IsTruncated>(.*?)</IsTruncated>", listing))
for k, mod, size in keys:
    print(f"{int(size):>12}  {mod}  {k}")
published = cai2.published_md5()

t = time.time()
streamed = {ind: cai2.stream_archive(ind) for ind in cai2.ARCHIVES}
print(f"streamed both archives in {time.time() - t:.0f} s")

In [ ]:
print("== checksums")
for ind, (recs, md5, nbytes, others) in streamed.items():
    path = cai2.ARCHIVES[ind]
    print(f"{path}: streamed {nbytes} bytes, MD5 {md5}, published {published[path]}, match {md5 == published[path]}; "
          f"{len(recs)} .mat files, other files {others}")

all_recs = [r for recs, *_ in streamed.values() for r in recs]
r0 = all_recs[0]
print("\n== contents of each file (first file shown; all files checked below)")
print("time series keys:", r0["keys"])
print("descriptions:", r0["descr"])
print("time units of the five series (2 = second):", sorted({tuple(r["time_units"]) for r in all_recs}),
      "; trialTimeUnit:", sorted({r["trial_time_unit"] for r in all_recs}))
print("indicator field by archive:", {ind: sorted({r["indicator"] for r in recs}) for ind, (recs, *_) in streamed.items()})
print("values of detected_spikes:", sorted({v for r in all_recs for v in r["spike_values"]}))
print("fluorescence and neuropil share a time base in every file:", all(r["tf_neuropil_same"] for r in all_recs))
print("raw, filtered ephys and spikes share a time base in every file:", all(r["ephys_same_clock"] for r in all_recs))
print("ephys sample interval (s), min and max over files:",
      min(r["ephys_dt"][0] for r in all_recs), max(r["ephys_dt"][1] for r in all_recs))

rows = []
for r in all_recs:
    isi = np.diff(r["spikes"])
    rows.append(dict(indicator=r["indicator"], file=r["file"], cell=r["cell"], date=r["date"],
                     frames=len(r["tf"]), fps=round(cai2.fps(r), 2), frame_t0=round(r["tf"][0], 3),
                     frame_t1=round(r["tf"][-1], 2), ephys_t0=round(r["ephys_t0"], 4), ephys_t1=round(r["ephys_t1"], 2),
                     spikes=len(r["spikes"]), rate_hz=round(len(r["spikes"]) / (r["ephys_t1"] - r["ephys_t0"]), 2),
                     min_isi_ms=round(isi.min() * 1000, 1) if len(isi) else np.nan,
                     roi=f"{r['roi'].min():.0f}-{r['roi'].max():.0f}", neuropil=f"{r['neuropil'].min():.0f}-{r['neuropil'].max():.0f}",
                     raw_ephys=f"{r['raw_ephys_range'][0]:.2f} to {r['raw_ephys_range'][1]:.2f}"))
table = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 100)
print(table.to_string(index=False))
print("\nframe rate range by indicator:", table.groupby("indicator")["fps"].agg(["min", "max"]).to_dict("index"))
print("ISIs under 1 ms:", {r["file"]: int((np.diff(r["spikes"]) < 0.001).sum()) for r in all_recs if (np.diff(r["spikes"]) < 0.001).any()})

### Recordings deposited twice

Files are compared by the MD5 of their soma fluorescence and of their raw ephys trace. A pair with
both identical is the same recording deposited twice, and only the first file of the pair is kept.
Files split into `_part1` and `_part2` are parts of one recording; their time ranges are checked
for overlap.

In [ ]:
groups = {}
for r in all_recs:
    groups.setdefault((r["md5_roi"], r["md5_raw_ephys"]), []).append(r["file"])
dups = [v for v in groups.values() if len(v) > 1]
print("identical recordings:", dups)
for k in {r["md5_roi"] for r in all_recs}:
    files = [r["file"] for r in all_recs if r["md5_roi"] == k]
    if len(files) > 1:
        same_member = len({r["md5_member"] for r in all_recs if r["md5_roi"] == k}) == 1
        print("  same fluorescence in", files, "; whole MATLAB files identical:", same_member)
drop = {f for v in dups for f in v[1:]}
recs = {ind: [r for r in rs if r["file"] not in drop] for ind, (rs, *_) in streamed.items()}
print("dropped as duplicates:", sorted(drop))

# Shared stretches between any two files: 100-frame windows of soma fluorescence with the same bytes.
win = {}
for r in all_recs:
    for s in range(0, len(r["roi"]) - 100, 100):
        win.setdefault(r["roi"][s:s + 100].tobytes(), set()).add(r["file"])
shared = {}
for files in win.values():
    if len(files) > 1:
        key = tuple(sorted(files)); shared[key] = shared.get(key, 0) + 1
print("pairs of files sharing identical 100-frame windows:", shared)

parts = [r for r in all_recs if r["part"]]
for r in sorted(parts, key=lambda r: (r["cell"], r["recording"], r["part"])):
    print(f"  {r['file']}: ephys {r['ephys_t0']:.1f} to {r['ephys_t1']:.1f} s, frames {r['tf'][0]:.1f} to {r['tf'][-1]:.1f} s")
for ind, rs in recs.items():
    print(f"{ind}: {len(rs)} recordings kept, {len({r['cell'] for r in rs})} cells, {len({r['date'] for r in rs})} recording dates, "
          f"{sum(len(r['spikes']) for r in rs)} spikes, {sum(r['ephys_t1'] - r['ephys_t0'] for r in rs) / 60:.1f} min of recording")

### Shared clock of spikes and fluorescence

The spikes (10 kHz ephys clock) and the fluorescence frames (frame times) are on one time base by
the description ("the times are relative session start"). This is checked per recording: the
correlation of the spike count in each frame with the frame-to-frame rise of the corrected
fluorescence, at lags of -15 to +15 frames. A shared clock puts the peak at lag 0 or +1. As a
null, the same correlation with the spike train circularly shifted by a random amount (at least
300 frames; 20 shifts, seed 0) gives the size expected without a relation.

In [ ]:
rng = np.random.default_rng(0)
LAGS = np.arange(-15, 16)
clock = []
for ind, rs in recs.items():
    for r in rs:
        F = cai2.corrected_f(r); tf = r["tf"]
        edges = np.concatenate([tf, [tf[-1] + np.median(np.diff(tf))]])
        cnt = np.histogram(r["spikes"], edges)[0].astype(float)[:-1]
        dF = np.diff(F)
        xc = np.array([np.corrcoef(np.roll(cnt, l), dF)[0, 1] for l in LAGS])
        null = [np.nanmax([np.corrcoef(np.roll(np.roll(cnt, rng.integers(300, len(cnt) - 300)), l), dF)[0, 1] for l in LAGS])
                for _ in range(20)]
        clock.append(dict(indicator=ind, file=r["file"], spikes=len(r["spikes"]), peak_lag=int(LAGS[np.nanargmax(xc)]),
                          r_peak=round(float(np.nanmax(xc)), 3), r_lag0=round(float(xc[LAGS == 0][0]), 3),
                          null_max=round(float(np.max(null)), 3)))
clock = pd.DataFrame(clock)
clock["above_null"] = clock.r_peak > clock.null_max
print(clock.to_string(index=False))
print("\npeak lag counts:", clock.peak_lag.value_counts().to_dict())
print("recordings whose peak is not above the shifted null:", clock.loc[~clock.above_null, "file"].tolist())

## Supplementary-file claims (data elsewhere)

Supplementary files 1 (RCaMP1h variants) and 2 (R-GECO1 variants) of the paper are streamed from
the eLife CDN. Each row is a variant; the screen values are relative to the parent.

In [ ]:
SUPP = {1: "https://cdn.elifesciences.org/articles/12727/elife-12727-supp1-v2.xlsx",   # doi:10.7554/eLife.12727.030
        2: "https://cdn.elifesciences.org/articles/12727/elife-12727-supp2-v2.xlsx"}   # doi:10.7554/eLife.12727.031
supp = {}
for i, url in SUPP.items():
    with fsspec.open(url, "rb") as f:
        raw = f.read()
    x = pd.read_excel(io.BytesIO(raw), header=1)
    x.columns = [str(c).strip() for c in x.columns]
    x = x.rename(columns={x.columns[0]: "variant", x.columns[2]: "mutations"})
    x["variant"] = x["variant"].astype(str).str.strip().str.strip('"')
    x["mutations"] = x["mutations"].astype(str).str.strip()
    supp[i] = x
    print(f"Supplementary file {i}: {len(raw)} bytes, {len(x)} rows; columns {list(x.columns)}")
    print("  first rows:", x["variant"].head(5).tolist())

### screen-single-mutations-improved

> In the neuronal culture screen, 353 of 934 single-mutation RCaMP1h variants and 187 of 689
> single-mutation R-GECO1 variants had a higher 1 AP dF/F0 amplitude than their parent.

A single-mutation variant is a row whose "Mutations added" column holds one mutation (the parent's
own row, marked "base", is not a mutation); improved is 1 AP dF/F0 ratio above 1 with 1 AP
p-value below 0.01 (criterion). Variants with other rules are printed for reference.

In [ ]:
cid = "screen-single-mutations-improved"
ex = expected(cid)
res = {}
for i, parent in ((1, "RCaMP1h"), (2, "R-GECO1")):
    x = supp[i]
    nmut = x["mutations"].str.split().str.len()
    single = x[(nmut == 1) & (x["mutations"] != "base")]
    a1, p1 = single["1 AP ∆F/F0"], single["1 AP p-value"]
    res[parent] = dict(single=len(single), improved=int(((a1 > 1) & (p1 < 0.01)).sum()))
    print(f"{parent}: rows {len(x)}, mutation-count distribution {nmut.value_counts().sort_index().to_dict()}")
    print(f"  single-mutation variants {len(single)}; improved (ratio > 1, p < 0.01) {res[parent]['improved']}")
    print("  variants:", {
        "ratio > 1 alone": int((a1 > 1).sum()),
        "ratio > 1, p < 0.05": int(((a1 > 1) & (p1 < 0.05)).sum()),
        "p < 0.01 either direction": int((p1 < 0.01).sum()),
        "ratio > 1, p < 0.01 at any of 1, 3, 10, 160 AP": int(np.any([(single[f"{n} AP ∆F/F0"] > 1) & (single[f"{n} AP p-value"] < 0.01)
                                                                    for n in (1, 3, 10, 160)], axis=0).sum()),
        "missing 1 AP values": int(a1.isna().sum())})
    allrows = x[x["mutations"] != "base"]
    print(f"  over all {len(allrows)} variant rows: improved {int(((allrows['1 AP ∆F/F0'] > 1) & (allrows['1 AP p-value'] < 0.01)).sum())}")
measured = [res["RCaMP1h"]["improved"], res["RCaMP1h"]["single"], res["R-GECO1"]["improved"], res["R-GECO1"]["single"]]
for m, e in zip(measured, ex):
    print(f"{e['quantity']}: measured {m}, expected {e['value']} +/- {e['tolerance']}, within {within(m, e)}")

**Result: not reproduced.** The tables hold 934 single-mutation RCaMP1h variants and 692 R-GECO1
variants, both within tolerance of the paper's 934 and 689, but by the paper's rule (1 AP ratio
above 1, p < 0.01) only 230 RCaMP1h and 58 R-GECO1 variants are improved, against 353 and 187.
No variant rule printed above reaches the paper's counts.

### culture-jrgeco1a-sensitivity

> In cultured neurons jRGECO1a has an 8.5-fold larger 1 AP dF/F0 amplitude than R-GECO1 and a
> faster rise time, with decay time and maximal dF/F0 similar to R-GECO1.

In [ ]:
cid = "culture-jrgeco1a-sensitivity"
row = supp[2].set_index("variant").loc["jRGECO1a"]
print(row.drop("mutations").to_string())
e = expected(cid)[0]
print(f"\n1 AP ratio {row['1 AP ∆F/F0']:.3f}, expected {e['value']} +/- {e['tolerance']}, within {within(row['1 AP ∆F/F0'], e)}")
for k, p in (("Decay half time (10 AP)", "Decay half time (10 AP) p-value"), ("160 AP ∆F/F0", "160 AP p-value")):
    print(f"{k}: ratio {row[k]:.3f} (in 0.8 to 1.25: {0.8 <= row[k] <= 1.25}), p {row[p]:.3g} (>= 0.01: {row[p] >= 0.01})")

**Result: reproduced.** The jRGECO1a row gives a 1 AP ratio of 8.57 against R-GECO1; its 10 AP
half decay time (1.05) and 160 AP dF/F0 (1.10) ratios are within 0.8 to 1.25, with p-values 0.32
and 0.47. The rise time is not in the table and is not tested.

### culture-jrcamp1-sensitivity

> jRCaMP1a and jRCaMP1b have 24-fold and 13-fold higher sensitivity than RCaMP1h for 1 AP stimuli
> in cultured neurons; jRCaMP1a is more sensitive and decays more slowly than jRCaMP1b.

In [ ]:
cid = "culture-jrcamp1-sensitivity"
t1 = supp[1].set_index("variant").loc[["jRCaMP1a", "jRCaMP1b"]]
print(t1.drop(columns="mutations").T.to_string())
for v, e in zip(["jRCaMP1a", "jRCaMP1b"], expected(cid)):
    print(f"{v} 1 AP ratio {t1.loc[v, '1 AP ∆F/F0']:.2f}, expected {e['value']} +/- {e['tolerance']}, within {within(t1.loc[v, '1 AP ∆F/F0'], e)}")
print("jRCaMP1a larger 1 AP ratio:", t1.loc["jRCaMP1a", "1 AP ∆F/F0"] > t1.loc["jRCaMP1b", "1 AP ∆F/F0"],
      "; larger 10 AP half decay ratio:", t1.loc["jRCaMP1a", "Decay half time (10 AP)"] > t1.loc["jRCaMP1b", "Decay half time (10 AP)"])

**Result: reproduced.** The 1 AP ratios are 24.02 (jRCaMP1a) and 12.96 (jRCaMP1b), and jRCaMP1a
has the larger 1 AP ratio and the longer 10 AP half decay (3.13 against 1.15 times RCaMP1h).

## Figure 4: simultaneous imaging and cell-attached recording

### fig4-recorded-cells

> Simultaneous imaging and cell-attached recording were made from 11 jRGECO1a cells in 6 mice and
> 10 jRCaMP1a cells in 5 mice.

Cells are the cell number and date in the file names; the data set does not name animals, so a
mouse is a recording date (criterion).

In [ ]:
cid = "fig4-recorded-cells"
measured_cells = []
for ind, rs in recs.items():
    cells = sorted({r["cell"] for r in rs}); dates = sorted({r["date"] for r in rs})
    measured_cells += [len(cells), len(dates)]
    print(f"{ind}: {len(cells)} cells on {len(dates)} dates")
    for d in dates:
        print(f"   {d}: cells {[c.split('_')[1] for c in cells if c.startswith(d)]}, recordings {sum(r['date'] == d for r in rs)}")
for m, e in zip(measured_cells, expected(cid)):
    print(f"{e['quantity']}: measured {m}, expected {e['value']}, match {within(m, e)}")

**Result: reproduced.** The file names give 11 jRGECO1a cells on 6 dates and 10 jRCaMP1a cells on 5
dates, as the paper reports. A mouse is identified by its recording date, which the data set does
not confirm; the criterion takes dates as mice unless animals are named, and they are not.

### fig4d-event-counts

> The Figure 4 analysis rests on 199, 70, 29, 34, 35, 22, 14 and 21 jRGECO1a events and 135, 65, 71,
> 52, 33, 20, 14 and 11 jRCaMP1a events of 1 to 8 APs in their time bins, and on 320 (jRGECO1a, 4 s)
> and 274 (jRCaMP1a, 5 s) no-AP noise events.

Isolation readings (a) more than 1 s before and after, (b) before only, and the data set's script
(1 s before, 0.8 s after, as `find_nspike_event.m` is called; noise as `find_no_event.m`). Noise
readings (i), (ii) and the script's. A combined reading pairs one isolation reading with one
noise reading; the claim counts how many of the 18 counts are within tolerance. The duplicate
jRCaMP1a recording is counted once.

In [ ]:
cid = "fig4d-event-counts"
ex = expected(cid)
ISO = ["a", "b", "script"]; NOISE = ["i", "ii", "script"]
events = {(ind, iso, n): [cai2.nspike_events(r, n, cai2.BIN_S[ind][n], iso) for r in rs]
          for ind, rs in recs.items() for iso in ISO for n in range(1, 9)}
noise = {(ind, nr): [cai2.noise_events(r, cai2.NOISE_S[ind], nr) for r in rs] for ind, rs in recs.items() for nr in NOISE}
count = lambda lst: int(sum(len(x) for x in lst))
exp_ev = {"jRGECO1a": ex[0:8], "jRCaMP1a": ex[8:16]}; exp_nz = {"jRGECO1a": ex[16], "jRCaMP1a": ex[17]}
tab = []
for ind in recs:
    for iso in ISO:
        c = [count(events[(ind, iso, n)]) for n in range(1, 9)]
        tab.append(dict(indicator=ind, reading=f"isolation {iso}", **{f"{n}AP": v for n, v in zip(range(1, 9), c)},
                        within=sum(within(v, e) for v, e in zip(c, exp_ev[ind]))))
    for nr in NOISE:
        v = count(noise[(ind, nr)])
        tab.append(dict(indicator=ind, reading=f"noise {nr}", noise=v, within=int(within(v, exp_nz[ind]))))
    tab.append(dict(indicator=ind, reading="paper", **{f"{n}AP": e["value"] for n, e in zip(range(1, 9), exp_ev[ind])}, noise=exp_nz[ind]["value"]))
print(pd.DataFrame(tab).to_string(index=False))

combo = []
for iso, nr in itertools.product(ISO, NOISE):
    k = 0
    for ind in recs:
        k += sum(within(count(events[(ind, iso, n)]), e) for n, e in zip(range(1, 9), exp_ev[ind]))
        k += within(count(noise[(ind, nr)]), exp_nz[ind])
    combo.append(dict(isolation=iso, noise=nr, within_of_18=int(k)))
combo = pd.DataFrame(combo).sort_values("within_of_18", ascending=False)
print("\ncounts within tolerance, of 18, per combined reading:\n" + combo.to_string(index=False))
measured_counts = ([count(events[("jRGECO1a", "script", n)]) for n in range(1, 9)] + [count(events[("jRCaMP1a", "script", n)]) for n in range(1, 9)]
                   + [count(noise[("jRGECO1a", "script")]), count(noise[("jRCaMP1a", "script")])])
print("\nscript reading, as measured:", measured_counts)
for m, e in zip(measured_counts, ex):
    print(f"  {e['quantity']}: {m} (expected {e['value']} +/- {e['tolerance']}) {'within' if within(m, e) else 'outside'}")

Breakdown by cell and recording date under the script reading, and two variants: the counts with
the duplicate jRCaMP1a file included, and, for jRCaMP1a, isolation intervals other than the
script's, to show how far the counts move with them (not a reading of the claim).

In [ ]:
by = []
for ind, rs in recs.items():
    for j, r in enumerate(rs):
        by.append(dict(indicator=ind, date=r["date"], cell=r["cell"], **{f"{n}AP": len(events[(ind, "script", n)][j]) for n in range(1, 9)},
                       noise=len(noise[(ind, "script")][j])))
by = pd.DataFrame(by)
print(by.groupby(["indicator", "date", "cell"]).sum(numeric_only=True).to_string())
print("\nper date:\n" + by.groupby(["indicator", "date"]).sum(numeric_only=True).to_string())

dup_recs = streamed["jRCaMP1a"][0]
print("\njRCaMP1a with the duplicate file included (script reading):",
      [sum(len(cai2.nspike_events(r, n, cai2.BIN_S["jRCaMP1a"][n], "script")) for r in dup_recs) for n in range(1, 9)],
      "noise", sum(len(cai2.noise_events(r, 5.0, "script")) for r in dup_recs))
sweep = []
for pre, post in itertools.product([1.0, 1.25, 1.5, 2.0], [0.8, 1.0, 1.5, 2.0]):
    cai2.ISOLATION["sweep"] = (pre, post)
    c = [sum(len(cai2.nspike_events(r, n, cai2.BIN_S["jRCaMP1a"][n], "sweep")) for r in recs["jRCaMP1a"]) for n in range(1, 9)]
    sweep.append(dict(pre_s=pre, post_s=post, counts=c, within_of_8=sum(within(v, e) for v, e in zip(c, exp_ev["jRCaMP1a"]))))
del cai2.ISOLATION["sweep"]
print("\njRCaMP1a event counts against isolation intervals (variant; spike on each side not required):")
print(pd.DataFrame(sweep).to_string(index=False))
print("noise counts for jRCaMP1a with script noise rule and bins of 4, 5, 6 s:",
      {L: sum(len(cai2.noise_events(r, L, "script")) for r in recs["jRCaMP1a"]) for L in (4.0, 5.0, 6.0)})

**Result: partially reproduced.** Under the data set's script reading all nine jRGECO1a counts
match the paper exactly (199, 70, 29, 34, 35, 22, 14, 21 events and 320 noise traces), but only
two of the nine jRCaMP1a counts are within tolerance (7 and 8 AP); the jRCaMP1a counts are higher
(162, 89, 86, 62, 44, 26 for 1 to 6 AP) and the noise count lower (246 against 274). That is 11 of
18, the best of the nine combined readings, in the 10 to 14 band. The paper's jRCaMP1a counts
are not produced by the script's rule; the isolation sweep finds no interval pair that matches
more than 7 of the 8 jRCaMP1a counts.

### Detection: traces, templates and projections (claims fig4e-* and fig4df)

The traces of every event set are cut once per segment reading. Segment readings (a), (b), (c)
are those of the claim; "script" is the data set's script (10 frames before to 20 after the
frame nearest the event, interpolated ten-fold, F0 from the 10 frames before, noise traces with
|dF/F0| >= 0.3 dropped). Templates (i) all events and (ii) leave-one-out. The detection rate is
the fraction of event projections above the 95th percentile of the noise projections (5% false
positives, Methods and criterion); the script's 99th percentile is shown as a variant.

The script is written for jRGECO1a at 30 Hz. Its frame window applied to jRCaMP1a at 15 Hz covers
0.67 s before to 1.33 s after the event, twice the time; that it was the jRCaMP1a analysis is an
assumption.

In [ ]:
SEGS = ["a", "b", "c", "script"]
t = time.time()
sig_tr, noise_tr, dropped = {}, {}, {}
for ind, rs in recs.items():
    for seg in SEGS:
        for nr in NOISE:
            parts = [cai2.segment_traces(r, ts, seg, ind, noise=True) for r, ts in zip(rs, noise[(ind, nr)])]
            noise_tr[(ind, nr, seg)] = np.concatenate([p[0] for p in parts if len(p[0])])
            dropped[(ind, "noise " + nr, seg)] = sum(p[1] for p in parts)
        for iso in ISO:
            for n in range(1, 9):
                parts = [cai2.segment_traces(r, ts, seg, ind) for r, ts in zip(rs, events[(ind, iso, n)])]
                sig_tr[(ind, iso, n, seg)] = np.concatenate([p[0] for p in parts if len(p[0])])
                dropped[(ind, f"{n}AP {iso}", seg)] = sum(p[1] for p in parts)

def after_spike(tr, seg):
    # the part of each trace after the first spike: frames for a, c; frames 10 on for b; samples 100 to 300 for script
    return tr[:, 10:] if seg == "b" else tr[:, 100:] if seg == "script" else tr

rows = []
for (ind, iso, n, seg), S in sig_tr.items():
    for nr in NOISE:
        N = noise_tr[(ind, nr, seg)]
        for tpl in ["all", "loo"]:
            ps, pn = cai2.projections(S, N, tpl)
            rows.append(dict(indicator=ind, nAP=n, isolation=iso, noise=nr, segment=seg, template=tpl, n_events=len(S), n_noise=len(N),
                             rate95=100 * cai2.detection_rate(ps, pn, 95), rate99=100 * cai2.detection_rate(ps, pn, 99),
                             balanced95=100 * cai2.balanced_accuracy(ps, pn, 95),
                             dprime_i=cai2.dprime(ps, pn, "i"), dprime_ii=cai2.dprime(ps, pn, "ii"),
                             median_peak_seg=float(np.median(after_spike(S, seg).max(1)))))
grid = pd.DataFrame(rows)
peak1s = {(ind, iso, n): np.concatenate([cai2.peak_dff(r, ts, ind, "1s") for r, ts in zip(rs, events[(ind, iso, n)])])
          for ind, rs in recs.items() for iso in ISO for n in range(1, 9)}
grid["median_peak_1s"] = [float(np.median(peak1s[(r.indicator, r.isolation, r.nAP)])) for r in grid.itertuples()]
print(f"{len(grid)} combinations in {time.time() - t:.0f} s")
dr = pd.Series(dropped)
print("traces dropped at recording edges (or, for script noise, by the 0.3 limit), nonzero only:")
print(dr[dr > 0].to_string())
MAIN = dict(isolation="script", noise="script", segment="script", template="all")
def pick(df, **kw):
    m = np.ones(len(df), bool)
    for k, v in kw.items():
        m &= df[k] == v
    return df[m]

### fig4e-jrgeco1a-1ap-detection

> With jRGECO1a, single spikes are detected with 82% accuracy at a 5% false-positive rate (11
> cells, 6 mice, 199 spikes).

In [ ]:
cid = "fig4e-jrgeco1a-1ap-detection"
ex = expected(cid)
for n, e in zip([1], ex):
    g = pick(grid, indicator="jRGECO1a", nAP=n)
    main = pick(g, **MAIN).iloc[0]
    print(f"== jRGECO1a {n} AP: main reading (script isolation, noise and segment, template from all events):"
          f" rate {main.rate95:.1f}% ({main.n_events} events, {main.n_noise} noise traces); expected {e['value']} +/- {e['tolerance']}:"
          f" within {within(main.rate95, e)}")
    print(f"   variants on the main reading: 99th percentile (script) {main.rate99:.1f}%, balanced accuracy {main.balanced95:.1f}%,"
          f" leave-one-out {pick(g, isolation='script', noise='script', segment='script', template='loo').iloc[0].rate95:.1f}%")
    named = g[g.segment != "script"]
    ok = named[[within(v, e) for v in named.rate95]]
    print(f"   claim's named segment readings (a, b, c): rate {named.rate95.min():.1f} to {named.rate95.max():.1f}%;"
          f" {len(ok)} of {len(named)} combinations within tolerance")
    print(named.pivot_table(index=["isolation", "noise"], columns=["segment", "template"], values="rate95").round(1).to_string())
    print("   script segment, all combinations of isolation and noise readings:")
    print(pick(g, segment="script").pivot_table(index=["isolation", "noise"], columns="template", values="rate95").round(1).to_string())

In [ ]:
# Main reading broken down by recording date (mouse): events of each date scored on the pooled template and noise threshold.
rs = recs["jRGECO1a"]
N = noise_tr[("jRGECO1a", "script", "script")]
for n in [1]:
    per = [(r["date"], cai2.segment_traces(r, ts, "script", "jRGECO1a")[0]) for r, ts in zip(rs, events[("jRGECO1a", "script", n)])]
    S = np.concatenate([tr for _, tr in per if len(tr)])
    dates = np.concatenate([[d] * len(tr) for d, tr in per if len(tr)])
    ps, pn = cai2.projections(S, N, "all")
    th = np.percentile(pn, 95)
    out = pd.DataFrame(dict(date=dates, hit=ps > th)).groupby("date").hit.agg(["size", "mean"])
    out["mean"] = (100 * out["mean"]).round(1)
    print(f"jRGECO1a {n} AP, detection rate by date (n events, % detected):", out.to_dict("index"))

**Result: reproduced.** Under the main reading (the data set's script for events, noise and
segment, the template from all events, the Methods' 5% false-positive threshold) 82% of the 199
single spikes are detected, the paper's value. Among the claim's named segment readings (a),
(b), (c) the rate runs from 0 to 76.3% with the noise and isolation readings, and 1 of 54
combinations is within tolerance (isolation (a), noise (ii), segment (b): 76.3%). The script's own
1% threshold gives 60.8%. By recording date the main-reading rate is 71.4 to 92.6% on the five
dates with 28 or more events.

### fig4e-jrgeco1a-2ap-detection

> With jRGECO1a, two APs within 100 ms are detected with 96% accuracy at a 5% false-positive rate
> (70 events).

In [ ]:
cid = "fig4e-jrgeco1a-2ap-detection"
ex = expected(cid)
for n, e in zip([2], ex):
    g = pick(grid, indicator="jRGECO1a", nAP=n)
    main = pick(g, **MAIN).iloc[0]
    print(f"== jRGECO1a {n} AP: main reading (script isolation, noise and segment, template from all events):"
          f" rate {main.rate95:.1f}% ({main.n_events} events, {main.n_noise} noise traces); expected {e['value']} +/- {e['tolerance']}:"
          f" within {within(main.rate95, e)}")
    print(f"   variants on the main reading: 99th percentile (script) {main.rate99:.1f}%, balanced accuracy {main.balanced95:.1f}%,"
          f" leave-one-out {pick(g, isolation='script', noise='script', segment='script', template='loo').iloc[0].rate95:.1f}%")
    named = g[g.segment != "script"]
    ok = named[[within(v, e) for v in named.rate95]]
    print(f"   claim's named segment readings (a, b, c): rate {named.rate95.min():.1f} to {named.rate95.max():.1f}%;"
          f" {len(ok)} of {len(named)} combinations within tolerance")
    print(named.pivot_table(index=["isolation", "noise"], columns=["segment", "template"], values="rate95").round(1).to_string())
    print("   script segment, all combinations of isolation and noise readings:")
    print(pick(g, segment="script").pivot_table(index=["isolation", "noise"], columns="template", values="rate95").round(1).to_string())

In [ ]:
# Main reading broken down by recording date (mouse): events of each date scored on the pooled template and noise threshold.
rs = recs["jRGECO1a"]
N = noise_tr[("jRGECO1a", "script", "script")]
for n in [2]:
    per = [(r["date"], cai2.segment_traces(r, ts, "script", "jRGECO1a")[0]) for r, ts in zip(rs, events[("jRGECO1a", "script", n)])]
    S = np.concatenate([tr for _, tr in per if len(tr)])
    dates = np.concatenate([[d] * len(tr) for d, tr in per if len(tr)])
    ps, pn = cai2.projections(S, N, "all")
    th = np.percentile(pn, 95)
    out = pd.DataFrame(dict(date=dates, hit=ps > th)).groupby("date").hit.agg(["size", "mean"])
    out["mean"] = (100 * out["mean"]).round(1)
    print(f"jRGECO1a {n} AP, detection rate by date (n events, % detected):", out.to_dict("index"))

**Result: reproduced.** Under the main reading 100% of the 70 two-spike events are detected,
inside the 92 to 100% the criterion allows; 15 of the 54 combinations of the claim's named
readings are also within it, and the script's 1% threshold gives 98.6%.

### fig4e-jrcamp1a-detection

> jRCaMP1a fluorescence changes are smaller, with 1 AP and 2 AP detection rates of 65% and 80% at a
> 5% false-positive rate (10 cells, 5 mice, 135 and 65 events).

In [ ]:
cid = "fig4e-jrcamp1a-detection"
ex = expected(cid)
for n, e in zip([1, 2], ex):
    g = pick(grid, indicator="jRCaMP1a", nAP=n)
    main = pick(g, **MAIN).iloc[0]
    print(f"== jRCaMP1a {n} AP: main reading (script isolation, noise and segment, template from all events):"
          f" rate {main.rate95:.1f}% ({main.n_events} events, {main.n_noise} noise traces); expected {e['value']} +/- {e['tolerance']}:"
          f" within {within(main.rate95, e)}")
    print(f"   variants on the main reading: 99th percentile (script) {main.rate99:.1f}%, balanced accuracy {main.balanced95:.1f}%,"
          f" leave-one-out {pick(g, isolation='script', noise='script', segment='script', template='loo').iloc[0].rate95:.1f}%")
    named = g[g.segment != "script"]
    ok = named[[within(v, e) for v in named.rate95]]
    print(f"   claim's named segment readings (a, b, c): rate {named.rate95.min():.1f} to {named.rate95.max():.1f}%;"
          f" {len(ok)} of {len(named)} combinations within tolerance")
    print(named.pivot_table(index=["isolation", "noise"], columns=["segment", "template"], values="rate95").round(1).to_string())
    print("   script segment, all combinations of isolation and noise readings:")
    print(pick(g, segment="script").pivot_table(index=["isolation", "noise"], columns="template", values="rate95").round(1).to_string())

In [ ]:
# Main reading broken down by recording date (mouse): events of each date scored on the pooled template and noise threshold.
rs = recs["jRCaMP1a"]
N = noise_tr[("jRCaMP1a", "script", "script")]
for n in [1, 2]:
    per = [(r["date"], cai2.segment_traces(r, ts, "script", "jRCaMP1a")[0]) for r, ts in zip(rs, events[("jRCaMP1a", "script", n)])]
    S = np.concatenate([tr for _, tr in per if len(tr)])
    dates = np.concatenate([[d] * len(tr) for d, tr in per if len(tr)])
    ps, pn = cai2.projections(S, N, "all")
    th = np.percentile(pn, 95)
    out = pd.DataFrame(dict(date=dates, hit=ps > th)).groupby("date").hit.agg(["size", "mean"])
    out["mean"] = (100 * out["mean"]).round(1)
    print(f"jRCaMP1a {n} AP, detection rate by date (n events, % detected):", out.to_dict("index"))

In [ ]:
# Variant: the main reading without the recordings whose fluorescence showed no relation to the spikes
# in the clock check (shifted null not exceeded).
bad = set(clock.loc[~clock.above_null, "file"])
ind = "jRCaMP1a"
keep = [j for j, r in enumerate(recs[ind]) if r["file"] not in bad]
N = np.concatenate([cai2.segment_traces(recs[ind][j], noise[(ind, "script")][j], "script", ind, noise=True)[0] for j in keep])
for n in (1, 2):
    S = np.concatenate([x for x in (cai2.segment_traces(recs[ind][j], events[(ind, "script", n)][j], "script", ind)[0] for j in keep) if len(x)])
    ps, pn = cai2.projections(S, N, "all")
    print(f"{ind} {n} AP without {sorted(bad)}: {len(S)} events, {len(N)} noise traces, rate {100 * cai2.detection_rate(ps, pn):.1f}%")

**Result: reproduced, resting on an assumption.** Under the main reading the jRCaMP1a 1 AP and
2 AP detection rates are 63.0% and 75.3%, both within tolerance, though on 162 and 89 events, not
the paper's 135 and 65 (see fig4d-event-counts). The main reading applies the jRGECO1a script's
frame window to jRCaMP1a, which is assumed. Under the claim's named segment readings (a), (b), (c)
no combination puts the 1 AP rate within tolerance (at most 41.6%), while 9 of 54 put the 2 AP
rate within it. The 1 AP rate differs between dates (85.7% on 20150605, 33.9% on 20150707).
Without the two recordings that failed the clock check, the rates are 75.4% (130 events) and
82.9% (76 events), the 1 AP rate then outside tolerance.

### fig4df-jrgeco1a-better-than-jrcamp1a

> jRGECO1a shows higher dF/F0 amplitudes and better spike detection (d-prime) than jRCaMP1a over
> the whole range of AP bursts from 1 to 8 APs.

Per n, the median peak dF/F0 (window (a) 1 s after the first spike, or (b) the projected
segment) and d-prime (readings (i) and (ii)), jRGECO1a against jRCaMP1a, under each combination
of readings; the paper reports no test.

In [ ]:
cid = "fig4df-jrgeco1a-better-than-jrcamp1a"
keys = ["isolation", "noise", "segment", "template", "nAP"]
G = grid[grid.indicator == "jRGECO1a"].set_index(keys); C = grid[grid.indicator == "jRCaMP1a"].set_index(keys)
cmp = pd.DataFrame(index=G.index)
for pk in ["median_peak_1s", "median_peak_seg"]:
    cmp[pk] = G[pk] > C[pk]
for dk in ["dprime_i", "dprime_ii"]:
    cmp[dk] = G[dk] > C[dk]
res = []
for (iso, nr, seg, tpl), sub in cmp.groupby(level=[0, 1, 2, 3]):
    for pk, dk in itertools.product(["median_peak_1s", "median_peak_seg"], ["dprime_i", "dprime_ii"]):
        both = (sub[pk] & sub[dk])
        res.append(dict(isolation=iso, noise=nr, segment=seg, template=tpl, peak=pk, dprime=dk, n_of_8=int(both.sum()),
                        peak_n=int(sub[pk].sum()), dprime_n=int(sub[dk].sum())))
res = pd.DataFrame(res)
print("values of n (of 8) where jRGECO1a exceeds jRCaMP1a in both, over", len(res), "combined readings:",
      res.n_of_8.value_counts().sort_index().to_dict())
m = pick(res, **MAIN)
print("\nmain reading:\n" + m.to_string(index=False))
show = ["nAP", "n_events", "median_peak_1s", "median_peak_seg", "dprime_i", "dprime_ii"]
print("\nmain reading, per n:")
print(pd.concat({"jRGECO1a": pick(grid, indicator="jRGECO1a", **MAIN)[show].set_index("nAP"),
                 "jRCaMP1a": pick(grid, indicator="jRCaMP1a", **MAIN)[show].set_index("nAP")}, axis=1).round(3).to_string())
print("\nreadings with all 8:", len(res[res.n_of_8 == 8]), "; with 6 or 7:", len(res[res.n_of_8.isin([6, 7])]))
print(res[res.n_of_8 == 8].groupby(["isolation", "noise", "segment"]).size().to_string())

In [ ]:
# Peak dF/F0 (1 s window) per n under the script isolation, by indicator, with quartiles, and the
# script's own amplitude (maximum of the median trace over samples 1 to 250).
q = []
for ind in recs:
    for n in range(1, 9):
        p = peak1s[(ind, "script", n)]
        S = sig_tr[(ind, "script", n, "script")]
        q.append(dict(indicator=ind, nAP=n, n=len(p), q1=np.percentile(p, 25), median=np.median(p), q3=np.percentile(p, 75),
                      min=p.min(), max=p.max(), script_amplitude=np.median(S, 0)[:250].max()))
q = pd.DataFrame(q).round(3)
print(q.to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3))
for ind, col in (("jRGECO1a", "tab:blue"), ("jRCaMP1a", "black")):
    d = q[q.indicator == ind]
    ax.plot(d.nAP, d["median"], "o-", color=col, label=ind)
    ax.fill_between(d.nAP, d.q1, d.q3, color=col, alpha=0.15)
ax.set_xlabel("APs in time bin"); ax.set_ylabel("peak dF/F0 (1 s window)"); ax.legend(); plt.tight_layout(); plt.show()
print("plotted: medians and quartiles; all values within the axes (range", q["q1"].min(), "to", q["q3"].max(), ")")

**Result: reproduced.** Under the main reading jRGECO1a has the higher median peak dF/F0 (both
windows) and the higher d-prime (both formulas) for all 8 values of n, for example 0.210 against
0.112 peak dF/F0 and d-prime 2.35 against 1.16 at 1 AP; 152 of the 288 combined readings give all
8, and the other 136 give 0 to 7 (the distribution is printed above).

### fig4g-jrgeco1a-faster

> jRGECO1a has faster rise and decay kinetics than jRCaMP1a for 2 AP responses.

2 AP events (100 ms and 150 ms bins) with no spike in the 2 s before, under each post-event
isolation reading; an exponential fitted from the peak; the 16 traces with the smallest RMS
residual kept. Half decay (a) tau ln 2, (b) peak to half peak on the trace; half rise from the
first spike to half peak.

In [ ]:
cid = "fig4g-jrgeco1a-faster"
kin_rows, kin = [], {}
for iso in ISO:
    post = cai2.ISOLATION[iso][1] or 1.0
    for ind, rs in recs.items():
        ks = []
        for r in rs:
            ts = cai2.nspike_events(r, 2, cai2.BIN_S[ind][2], iso, pre_override=2.0)
            for d in cai2.kinetics(r, ts, ind, post):
                d.update(date=r["date"]); ks.append(d)
        ks = pd.DataFrame(ks).sort_values("rms")
        kin[(iso, ind)] = ks
        for sel, sub in (("best 16", ks.head(16)), ("all fitted", ks)):
            row = dict(isolation=iso, indicator=ind, traces=sel, n=len(sub), available=len(ks))
            for k in ["rise", "decay_fit", "decay_trace"]:
                v = sub[k].dropna()
                row[k] = f"{1000 * v.mean():.0f} +/- {1000 * v.std(ddof=1) / np.sqrt(len(v)):.0f} ms (n={len(v)})"
            row["tau_at_bound"] = int((sub.tau >= 19.9).sum())
            kin_rows.append(row)
print(pd.DataFrame(kin_rows).to_string(index=False))
print("\njRGECO1a faster (mean of the best 16) by reading:")
for iso in ISO:
    g, c = kin[(iso, "jRGECO1a")].head(16), kin[(iso, "jRCaMP1a")].head(16)
    print(f"  isolation {iso}: rise {g.rise.mean() < c.rise.mean()}, decay (a) fit {g.decay_fit.mean() < c.decay_fit.mean()},"
          f" decay (b) trace {g.decay_trace.mean() < c.decay_trace.mean()}")
print("\nbest 16 by recording date:", {f"{iso} {ind}": kin[(iso, ind)].head(16).date.value_counts().to_dict()
                                      for iso in ISO for ind in recs})

In [ ]:
# Variant: a longer fit window, to 2 s after the first spike (past the isolation interval), with the same 16-trace rule.
for iso in ISO:
    out = {}
    for ind, rs in recs.items():
        ks = []
        for r in rs:
            ts = cai2.nspike_events(r, 2, cai2.BIN_S[ind][2], iso, pre_override=2.0)
            ks += cai2.kinetics(r, ts, ind, 2.0 - cai2.BIN_S[ind][2])
        out[ind] = pd.DataFrame(ks).sort_values("rms").head(16)
    g, c = out["jRGECO1a"], out["jRCaMP1a"]
    print(f"2 s window, isolation {iso}: rise {1000 * g.rise.mean():.0f} vs {1000 * c.rise.mean():.0f} ms, decay fit {1000 * g.decay_fit.mean():.0f} vs"
          f" {1000 * c.decay_fit.mean():.0f} ms, decay trace {1000 * g.decay_trace.mean():.0f} vs {1000 * c.decay_trace.mean():.0f} ms (jRGECO1a vs jRCaMP1a)")
# Variant: the 16 traces with the smallest RMS residual relative to their peak.
for iso in ISO:
    for ind in recs:
        ks = kin[(iso, ind)]
        b = ks.assign(rel=ks.rms / ks.peak).sort_values("rel").head(16)
        print(f"isolation {iso} {ind}: rise {1000 * b.rise.mean():.0f} ms, decay fit {1000 * b.decay_fit.mean():.0f} ms,"
              f" decay trace {1000 * b.decay_trace.mean():.0f} ms (n={b.decay_trace.notna().sum()})")

**Result: reproduced.** Under the script and (a) isolation readings the 16 best-fitted jRGECO1a
2 AP traces have a shorter mean half rise (19 against 50 ms, script reading) and a shorter mean
half decay, by the fit (172 against 969 ms) and on the trace (135 against 176 ms), than jRCaMP1a.
Under reading (b), without isolation after the event, the half rise is longer for jRGECO1a (262
against 129 ms) while both decays are shorter. The fitted jRCaMP1a decay is unstable (s.e.m. 357
to 1115 ms, and one fit at the 20 s bound under reading (a)). The half rise times of 15 to 50 ms
under readings (a) and script are shorter than one frame for both indicators (33 and 67 ms), so
they rest on the interpolation with the frame before the spike. With the fit window extended to
2 s after the first spike, jRGECO1a is faster on all three measures under every isolation reading
(script reading: half rise 17 against 68 ms, trace half decay 145 against 259 ms).

## Summary

| Claim | Outcome |
|---|---|
| screen-single-mutations-improved | not reproduced (230 and 58 improved single mutants, against 353 and 187) |
| culture-jrgeco1a-sensitivity | reproduced (data elsewhere) |
| culture-jrcamp1-sensitivity | reproduced (data elsewhere) |
| fig4-recorded-cells | reproduced |
| fig4d-event-counts | partially reproduced (jRGECO1a counts exact under the data set's script; jRCaMP1a counts differ) |
| fig4e-jrgeco1a-1ap-detection | reproduced |
| fig4e-jrgeco1a-2ap-detection | reproduced |
| fig4e-jrcamp1a-detection | reproduced, resting on an assumption |
| fig4df-jrgeco1a-better-than-jrcamp1a | reproduced |
| fig4g-jrgeco1a-faster | reproduced |
| fig4g-kinetics-like-gcamp6 | blocked (other) |
| the other 24 claims | blocked (data missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the detection claims use the frame window, the noise
traces and the event isolation of the data set's script, which the paper's Methods do not give,
with the Methods' 5% false-positive threshold in place of the script's 1%; the script is written
for jRGECO1a and its frame window is assumed for jRCaMP1a. The claim's named segment readings give
a wide range of rates. The kinetics fit window and the fit-error rule are choices the paper leaves
open.

What was found about the dataset: two jRCaMP1a files (`data_20150526_cell2_001.mat` and
`_002.mat`) are the same recording deposited twice; the data set names no animals, so mice are
counted by recording date; the visual stimulus is not included, as the description says. In two
jRCaMP1a recordings (`data_20150707_cell1_001.mat`, 304 spikes, and `data_20150605_cell7_003.mat`,
15 spikes) the fluorescence shows no relation to the spikes above a shifted-spike null, so their
shared clock could not be established. The streamed archives match the published MD5 sums.

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024:.0f} MB")